# 5. Example: DataAssimBench ensemble data assimilation using the NeuralGCM

The NeuralGCM takes input from an ERA5-like format and returns a forecast at roughly 1-hour intervals. Here we use an initial ensemble drawn from ERA5 and assimilate a single observation sampled from the ERA5 'truth'.

#### Get required packages

In [ ]:
%%capture

import sys

# Make sure that the gcsfs package is installed in the current Jupyter kernel
!{sys.executable} -m pip install gcsfs

# Make sure the dinosaur dynamical core is installed for NeuralGCM
!{sys.executable} -m pip install dinosaur

In [ ]:
import dabench as dab
from datetime import datetime
import xarray as xr
import os
import matplotlib.pyplot as plt

#### Generate an initial ensemble

In [ ]:
# Point to the location where the ERA5 dataset is stored on GCP
gcp_era5_path = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"

# Specify the desired lagged ensemble sampling strategy
sample_strategy = 'multi_year'

# Specify the target date to which the ensemble will be recentered
target_date = "20240921Z0"

# Specify the desired ensemble size
ensemble_size = 3

# Store locally or on cloud via zarr
output_destination = f'./zarr_{target_date}_{ensemble_size}mem'

# Only generate if you have not yet processed this date
if not os.path.exists(output_destination):
    # Set up the input arguments
    input_args = {
        'date_format': "%Y%m%dZ%H",
        'atmosphere_ensemble_s3_key': output_destination,
        'target_date': datetime.strptime(target_date,"%Y%m%dZ%H"),
        'sample_strategy': sample_strategy,
        'start_date': datetime.strptime(target_date,"%Y%m%dZ%H"),
        'ensemble_size': ensemble_size,
        'era5_path': gcp_era5_path,
    }
    
    # Generate the initial ensemble, and store to a zarr file
    dab.dasupport.GenEra5Ens(**input_args)

#### Check the initial ensemble

In [ ]:
# Open the ensemble dataset stored in zarr format as an xarray dataset
ds = xr.open_dataset(output_destination, engine='zarr')
ds

#### Test run the NeuralGCM model

In [ ]:
# Store the ensemble mean to use as input to the NeuralGCM
output_destination_mean = f'{output_destination}_mean'
ds.mean(dim='member').to_zarr(output_destination_mean, mode='w')

In [ ]:
model_input_args = {
    'era5_path': output_destination_mean,
    'atm_res': '2_8',
    'forecast_hours': 1,
    'data_stride': 1,
    'start_time': datetime.strptime(target_date,"%Y%m%dZ%H"),
    'inner_steps': 1,
    'outer_steps': 1,
}
model = NeuralGCM(params=model_input_args)

In [ ]:
# Run test forecast
model.full_sequence()

## Apply data assimilation

#### Set up the ETKF object

In [ ]:
# Pass the NeuralGCM model object into the DA cycler
da_input_args = {
    system_dim: ,
    ensemble_dim: ensemble_size,
    delta_t: 3600,
    model_obj: model,
    H: ,
    h: ,
}
etkf = dab.ETKF(**da_input_args)

#### Perform a DA cycle

In [ ]:
etkf_input_args = {
    input_state: ,
    start_time: target_date,
    obs_vector: ,
    n_cycles: 2,
    obs_error_sd: 1.0, #hPa
    analysis_window: 3600,
    return_forecast=True,
}
etkf.cycle(**etkf_input_args)